# Create and Upload I14Y Concept

## Codelist Entries

In [86]:
import json

# read JSON file to dict
with open('../data/output/parties.json', 'r') as file:
    parties = json.load(file)

In [87]:
import pandas as pd

# base für codes json
codes = {
    "data": [],
}

for id, party in parties.items():

    # only identity is relevant
    party = party.get("identity")

    if party.get("name") and pd.isna(party.get("dissolution_date")):  # check if the party has a name and is not deprecated
        code_entry = {
            "annotations": [
                {
                    "type": "EXT_RESOURCE",
                    "uri": f"https://politics.ld.admin.ch/political-party/{id}"
                }
            ],
            "code": str(id),  # get the id of the party
            "name": party.get("name")
        }
        child_of = [relation for relation in party.get("relationships", []) if relation.get("type_label") == "child-of"]
        if len(child_of) > 0:
            child_of = child_of[0]  # get the first child_of relation
            code_entry["parentCode"] = str(child_of.get("party_to"))
        codes["data"].append(code_entry)

# write the dict to a JSON file
with open('../data/output/i14y_values.json', 'w', encoding='utf-8') as file:
    json.dump(codes, file, indent=2, ensure_ascii=False)

## Check if Changes Exist

### Get Current Concept from I14Y public API

In [95]:
import requests
# get the current code list
url = f"https://api-a.i14y.admin.ch/api/public/v1/concepts/"

params = {
    'conceptIdentifier': 'political-parties', 
}

response = requests.get(url, params=params, verify = False)

data = response.json()

print(f"Response: {response.status_code}")

# sort data by 'version' in descending order
data_sorted = sorted(data["data"], key=lambda x: x['version'], reverse=True)
data = {"data": data_sorted[0]}
# save the latest concept to a JSON file
with open('../data/output/i14y_concept_current.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Response: 200


### Get current code list values

In [96]:
update = False

url = f"https://api-a.i14y.admin.ch/api/public/v1/concepts/{current_concept_id}/codelist-entries/exports/json"

response = requests.get(url, verify = False)

if response.status_code == 200:
    current_code_list_entries = json.loads(response.content)
else:
    print(f"Failed to get data, status code: {response.status_code}")

current_codes = set(json.dumps(d, sort_keys=True) for d in current_code_list_entries["data"])
new_codes = set(json.dumps(d, sort_keys=True) for d in codes["data"])

if current_codes == new_codes:
    print("No changes detected. The code list is up to date.")
else:
    print("Changes detected. The code list needs to be updated.")
    update = True


Changes detected. The code list needs to be updated.


/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Upload

In [90]:
from dotenv import load_dotenv

ENV = "INT"

# Load the env file
load_dotenv(f"../.env.{ENV.lower()}")

True

### Get Token

In [98]:
import requests
import os

def get_access_token(client_id,client_secret):
    data = {'grant_type': 'client_credentials'}
    response = requests.post("https://identity.i14y.a.c.bfs.admin.ch/realms/bfs-sis-a/protocol/openid-connect/token", data=data, verify=False, auth=(client_id, client_secret))
    return response.json()

if update:
    access_token = 'Bearer ' + get_access_token(os.getenv('I14Y_CLIENT_ID'), os.getenv('I14Y_CLIENT_SECRET'))['access_token']

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'identity.i14y.a.c.bfs.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### Set End Date of the Current Concept

In [ ]:
import requests
from datetime import datetime

# read template and current concept data

with open('../data/output/i14y_concept_template.json', 'r') as f:
    concept_template = json.load(f)

with open('../data/output/i14y_concept_current.json', 'r') as f:
    concept_current = json.load(f)

concept_template["data"]["validFrom"] = concept_current["data"]["validFrom"]
concept_template["data"]["validTo"] = datetime.now().isoformat()
concept_template["data"]["version"] = concept_current["data"]["version"]

headers = {
    'Content-Type': 'application/json',  # To make a valid request it's necessary to state the correct 'Content-Type' (Note: in the other POST APIs [point 6 and 7: import codelist entries] there is not need to state the 'Content-Type', in fact stating it it could lead to errors)
    'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    'Authorization': access_token
}

url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{concept_current["data"]["id"]}'

if update:

    response = requests.put(url, headers=headers, data=json.dumps(concept_template), verify=False)

    if response.status_code == 204:
        print(f'Status-Code: {response.status_code}')
    else:
        print(f"Error: {response.status_code} - {response.text}")

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Error: 204 - 


### Create Concept

In [100]:
import requests
from datetime import datetime

headers = {
    'Content-Type': 'application/json',  # To make a valid request it's necessary to state the correct 'Content-Type' (Note: in the other POST APIs [point 6 and 7: import codelist entries] there is not need to state the 'Content-Type', in fact stating it it could lead to errors)
    'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    'Authorization': access_token
}

url = 'https://api-a.i14y.admin.ch/api/partner/v1/concepts'

# get current date in ISO 8601 format

concept_template["data"]["validFrom"] = datetime.now().isoformat()
version_date = datetime.now()
version_date = f"{version_date.year}.{version_date.month}.{version_date.day}"
concept_template["data"]["version"] = version_date

# remove the "validTo" field from the concept_template before sending the POST request
if "validTo" in concept_template["data"]:
    del concept_template["data"]["validTo"]

if update:

    response = requests.post(url, headers=headers, data=json.dumps(concept_template), verify=False)

    if response.status_code == 201:
        print(f'Status-Code: {response.status_code}')
        concept_id = response.json()
        print(f'Concept Id: {concept_id}')
    else:
        print(f"Error: {response.status_code} - {response.text}")

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 201
Concept Id: 01a0d881-4b63-77d0-9f4b-6623d7173cd3


### Add Codelist Entries

In [101]:
headers = { 
    'Authorization': access_token,
    'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
}

url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{concept_id}/codelist-entries/imports/json'

file_path = '../data/output/i14y_values.json' #state the right file name here

if update:
    with open(file_path, 'rb') as file:
        files = {
            'file': (file_path, file, 'text/json')
        }
        response = requests.post(url, headers=headers, files=files, verify = False)

    print(f'Status-Code: {response.status_code} - {response.text}')
    print(response)


/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 204 - 
<Response [204]>


### Set Concept to Public

In [102]:
headers = { 
    'Authorization': access_token,
    'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
}

params = {
    'conceptId': concept_id,
    'level': "Public" 
}

url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{concept_id}/publication-level'


if update:
    response = requests.put(url, headers=headers, params=params)

    print(f'Status-Code: {response.status_code} - {response.text}')
    print(response)


Status-Code: 204 - 
<Response [204]>
